# Predicting arrears next month
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), walk-forward CV on training rows, choice by mean CV ROC AUC, holdout scored once.

**Feature decision:** `final_notice_sent` is recorded in the billing cycle *after* the billing month, i.e. after prediction time and close to the outcome (86% arrears when 1 vs 1.4% when 0). It is excluded as leakage. `meter_ref` and `billing_month` are excluded per the protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd, yaml
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
P=yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()==P['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()==P['split']['holdout_sha256']
df=pd.read_csv('data.csv'); assert len(df)==P['dataset']['rows']
hold=pd.read_csv('holdout.csv')['row'].values; assert len(hold)==P['split']['holdout_rows']
is_hold=df.index.isin(hold)
train=df[~is_hold]; test=df[is_hold]
print(len(train),len(test),train.billing_month.max(),test.billing_month.min())

In [ ]:
TARGET='arrears_next_month'
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']  # final_notice_sent excluded: recorded after prediction time
CAT=['tariff']; NUM=[f for f in FEATURES if f not in CAT]
def prep(scale):
    num=[('imp',SimpleImputer(strategy='median'))]+([('sc',StandardScaler())] if scale else [])
    return ColumnTransformer([('num',Pipeline(num),NUM),('cat',OneHotEncoder(handle_unknown='ignore'),CAT)])
# all candidates named before any is scored
CANDS={
 'baseline_prior':Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1':Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'rf_200_leaf20':Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=200,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_depth3':Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(CANDS))

In [ ]:
# walk-forward CV over training months: 5 splits, expanding window, validation = consecutive month blocks
months=np.sort(train.billing_month.unique()); n=5
k=len(months)//(n+1); edges=[k*(i+1) for i in range(n)]+[len(months)]
folds=[]
for i in range(n):
    trm=months[:edges[i]]; vam=months[edges[i]:edges[i+1]]
    folds.append((train.index[train.billing_month.isin(trm)].values, train.index[train.billing_month.isin(vam)].values))
for tr,va in folds: print(len(tr),len(va),df.billing_month[tr].max(),df.billing_month[va].min(),df.billing_month[va].max())

In [ ]:
cv_scores={}
for name,m in CANDS.items():
    s=[]
    for tr,va in folds:
        mm=clone(m).fit(df.loc[tr,FEATURES],df.loc[tr,TARGET])
        s.append(float(roc_auc_score(df.loc[va,TARGET],mm.predict_proba(df.loc[va,FEATURES])[:,1])))
    cv_scores[name]=s
res=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T; print(res)
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k])); print('chosen',chosen)

In [ ]:
final=clone(CANDS[chosen]).fit(train[FEATURES],train[TARGET])
holdout_auc=float(roc_auc_score(test[TARGET],final.predict_proba(test[FEATURES])[:,1]))
print('Holdout ROC AUC (scored once):',holdout_auc)

In [ ]:
ev={'features':FEATURES,'candidates':list(CANDS),
 'cv_folds':[[[int(x) for x in tr],[int(x) for x in va]] for tr,va in folds],
 'cv_scores':cv_scores,'chosen':chosen,
 'holdout_rows':[int(x) for x in hold],'holdout_scores':{chosen:holdout_auc}}
json.dump(ev,open('evidence.json','w'))